# RQ1 Dietary Habits and Obesity Classification

**Research question:** How do dietary habits influence obesity classification?

This Kaggle notebook loads the raw dataset, generates the publication-ready table as CSV, saves the figure as PDF, and prints the evidence-based answer.

In [ ]:

# ============================================================
# Kaggle-ready setup: load raw obesity dataset and shared helpers
# ============================================================
import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import chi2_contingency, pearsonr, spearmanr, f_oneway

# Kaggle output directory
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Change this only if needed. On Kaggle, the dataset usually lives under /kaggle/input/...
DATASET_PATH = None


def find_dataset_file():
    """Find the obesity dataset on Kaggle or locally. Supports CSV and Excel."""
    if DATASET_PATH:
        return DATASET_PATH
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './*.csv', './*.xlsx', './*.xls',
        '../input/**/*.csv', '../input/**/*.xlsx', '../input/**/*.xls'
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    # Prefer files whose name looks like the obesity dataset
    ranked = sorted(candidates, key=lambda p: ('obesity' not in os.path.basename(p).lower(), len(p)))
    if not ranked:
        raise FileNotFoundError('No CSV/XLSX dataset file found. Upload the raw dataset to Kaggle Input or set DATASET_PATH manually.')
    return ranked[0]


def load_dataset():
    path = find_dataset_file()
    print(f'Loading dataset from: {path}')
    if path.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


def clean_dataset(df):
    """Basic cleaning only; avoids changing the research meaning of variables."""
    df = df.copy()
    df = df.drop_duplicates().reset_index(drop=True)
    # Normalize object columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def add_obesity_group(df):
    """Collapse detailed target classes into paper-friendly groups."""
    df = df.copy()
    mapping = {
        'Insufficient_Weight': 'Underweight',
        'Normal_Weight': 'Normal',
        'Overweight_Level_I': 'Overweight',
        'Overweight_Level_II': 'Overweight',
        'Obesity_Type_I': 'Obese',
        'Obesity_Type_II': 'Obese',
        'Obesity_Type_III': 'Obese'
    }
    df['Obesity_Group'] = df['NObeyesdad'].map(mapping).fillna(df['NObeyesdad'])
    order = ['Underweight', 'Normal', 'Overweight', 'Obese']
    df['Obesity_Group'] = pd.Categorical(df['Obesity_Group'], categories=order, ordered=True)
    return df


def add_obesity_score(df):
    """Ordinal score for correlation/ANOVA summaries."""
    df = df.copy()
    score_map = {
        'Insufficient_Weight': 0,
        'Normal_Weight': 1,
        'Overweight_Level_I': 2,
        'Overweight_Level_II': 3,
        'Obesity_Type_I': 4,
        'Obesity_Type_II': 5,
        'Obesity_Type_III': 6
    }
    df['Obesity_Score'] = df['NObeyesdad'].map(score_map)
    return df


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved table: {path}')
    return path


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, format='pdf', bbox_inches='tight')
    print(f'Saved figure: {path}')
    plt.show()
    return path


def percent_yes(series):
    return (series.astype(str).str.lower().eq('yes').mean() * 100)


def style_plot(title, xlabel=None, ylabel=None):
    plt.title(title, fontsize=14, weight='bold')
    if xlabel: plt.xlabel(xlabel, fontsize=11)
    if ylabel: plt.ylabel(ylabel, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis='y', alpha=0.25)
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)


df = add_obesity_score(add_obesity_group(clean_dataset(load_dataset())))
display(df.head())
print(df['NObeyesdad'].value_counts())


## Analysis, table, figure, and answer

In [ ]:

# RQ1: Dietary habits and obesity classification
# Variables: FAVC, FCVC, NCP, CAEC, NObeyesdad

summary = (
    df.groupby('Obesity_Group', observed=False)
      .agg(
          sample_size=('NObeyesdad', 'size'),
          avg_vegetable_consumption_FCVC=('FCVC', 'mean'),
          avg_meals_per_day_NCP=('NCP', 'mean'),
          high_calorie_food_yes_percent=('FAVC', percent_yes)
      )
      .reset_index()
)
summary = summary.round(3)
save_table(summary, 'RQ1_dietary_habits_by_obesity_group.csv')
display(summary)

# Figure: normalized grouped bar chart for comparison across different units
plot_df = summary.copy()
for col in ['avg_vegetable_consumption_FCVC', 'avg_meals_per_day_NCP', 'high_calorie_food_yes_percent']:
    max_val = plot_df[col].max()
    plot_df[col + '_scaled'] = plot_df[col] / max_val if max_val != 0 else plot_df[col]

x = np.arange(len(plot_df))
width = 0.25
plt.figure(figsize=(9, 5.5))
plt.bar(x - width, plot_df['avg_vegetable_consumption_FCVC_scaled'], width, label='Vegetable intake, scaled')
plt.bar(x, plot_df['avg_meals_per_day_NCP_scaled'], width, label='Meals/day, scaled')
plt.bar(x + width, plot_df['high_calorie_food_yes_percent_scaled'], width, label='High-calorie food %, scaled')
plt.xticks(x, plot_df['Obesity_Group'].astype(str), rotation=15, ha='right')
style_plot('Dietary Habits Across Obesity Groups', 'Obesity group', 'Scaled value')
plt.legend(frameon=False, fontsize=9)
save_figure('RQ1_dietary_habits_by_obesity_group.pdf')

# Actual answer
highest_favc = summary.loc[summary['high_calorie_food_yes_percent'].idxmax(), 'Obesity_Group']
highest_fcvc = summary.loc[summary['avg_vegetable_consumption_FCVC'].idxmax(), 'Obesity_Group']
print('\nANSWER TO RQ1')
print(f"The group with the highest percentage of frequent high-calorie food consumption is: {highest_favc}.")
print(f"The group with the highest average vegetable consumption is: {highest_fcvc}.")
print('Overall, the table and figure show how dietary behavior varies across obesity classifications.')
